In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Files
# alt = file currently shown on the website (incl. manual corrections)
# neu = new output of markets_dataProcessing.ipynb

markets_old = Path("../public/markets_2026.csv")
markets_new = Path("markets_neu.csv")

In [3]:
KEY_COLS = ["id"]
IGNORE_COLS = []   # z.B. ["updated_at", "last_modified_by"]

# columns that are maintained by hand -> a difference here is usually a manual correction,
# a difference in the other (API) columns means the organiser changed the data
MANUAL_COLS = ["shortname", "Mo", "Di", "Mi", "Do", "Fr", "Sa", "So", "closed-exc", "hours-exc",
               "bemerkungen_en", "ignore", "immer-kostenlos", "international", "barrierefrei",
               "action", "image", "urheberschaft"]

# read everything as text, so "0" and 0 or empty and NaN are not reported as changes
CSV_READ_KW = dict(
    sep=",",
    encoding="utf-8",
    dtype=str,
    keep_default_na=False
)

In [4]:
def normalize_dataframe(df: pd.DataFrame, key_cols, ignore_cols):
    """
    Vereinheitlicht Spaltenreihenfolge (Keys zuerst),
    entfernt ignorierte Spalten, trimmt Strings.
    """
    df = df.copy()
    # Trim in Stringspalten
    for c in df.select_dtypes(include=["object","string"]).columns:
        df[c] = df[c].astype("string").str.strip()
    # Ignorierte Spalten entfernen
    drop_cols = [c for c in ignore_cols if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    # Spaltenreihenfolge: Keys zuerst
    rest = [c for c in df.columns if c not in key_cols]
    cols = list(key_cols) + rest
    return df[cols]

In [5]:
def compare_dataframes(df_old, df_new, key_cols):


    # Outer-Join auf Schlüssel, um Added/Removed zu erkennen
    merged_index = pd.merge(
        df_old[key_cols].drop_duplicates(), 
        df_new[key_cols].drop_duplicates(),
        on=key_cols, how="outer", indicator=True
    )
    added_keys = merged_index.loc[merged_index["_merge"] == "right_only", key_cols]
    removed_keys = merged_index.loc[merged_index["_merge"] == "left_only", key_cols]
    common_keys = merged_index.loc[merged_index["_merge"] == "both", key_cols]

    # Added / Removed Zeilen
    added_rows = added_keys.merge(df_new, on=key_cols, how="left")
    removed_rows = removed_keys.merge(df_old, on=key_cols, how="left")

    # Modified Zellen
    if common_keys.empty:
        modified_cells = pd.DataFrame(columns=key_cols + ["column","old","new"])
        return added_rows, removed_rows, modified_cells

    old_common = common_keys.merge(df_old, on=key_cols, how="left")
    new_common = common_keys.merge(df_new, on=key_cols, how="left")

    # Spalten vergleichen (alle außer Keys)
    value_cols = [c for c in df_new.columns if c not in key_cols]
    value_cols = [c for c in value_cols if c in old_common.columns]

    diffs = []
    for col in value_cols:
        a = old_common[col]
        b = new_common[col]
        neq = ~((a == b) | (a.isna() & b.isna()))
        if neq.any():
            diff_part = pd.concat([
                common_keys.reset_index(drop=True), 
                pd.Series([col]*len(common_keys), name="column"),
                a.reset_index(drop=True).rename("old"),
                b.reset_index(drop=True).rename("new")
            ], axis=1)
            modified = diff_part[neq.values]
            diffs.append(modified)

    modified_cells = pd.concat(diffs, ignore_index=True) if diffs else pd.DataFrame(columns=key_cols + ["column","old","new"])
    return added_rows, removed_rows, modified_cells

In [6]:
# Daten laden und normalisieren

df_old = pd.read_csv(markets_old, **CSV_READ_KW)
df_new = pd.read_csv(markets_new, **CSV_READ_KW)

# Validierung
missing_keys_old = [k for k in KEY_COLS if k not in df_old.columns]
missing_keys_new = [k for k in KEY_COLS if k not in df_new.columns]
if missing_keys_old or missing_keys_new:
    raise ValueError(f"Fehlende Schlüsselspalten – alt: {missing_keys_old}, neu: {missing_keys_new}")

# Normalisieren
df_old_n = normalize_dataframe(df_old, KEY_COLS, IGNORE_COLS)
df_new_n = normalize_dataframe(df_new, KEY_COLS, IGNORE_COLS)

print("Alt-Shape:", df_old_n.shape, "| Neu-Shape:", df_new_n.shape)
print("Spalten:", list(df_new_n.columns))

Alt-Shape: (47, 32) | Neu-Shape: (48, 32)
Spalten: ['id', 'name', 'shortname', 'strasse', 'bezirk', 'plz_ort', 'von', 'bis', 'oeffnungszeiten', 'Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So', 'closed-exc', 'hours-exc', 'w3', 'bemerkungen', 'bemerkungen_en', 'lat', 'lng', 'ignore', 'immer-kostenlos', 'international', 'barrierefrei', 'action', 'short_distance', 'train', 'image', 'urheberschaft']


In [7]:
# Vergleich durchführen

added_rows, removed_rows, modified_cells = compare_dataframes(df_old_n, df_new_n, KEY_COLS)

# market name and origin of the column (manual correction or API data)
modified_cells = modified_cells.merge(df_new_n[KEY_COLS + ["name"]], on=KEY_COLS, how="left")
modified_cells["herkunft"] = modified_cells["column"].map(lambda c: "manuell" if c in MANUAL_COLS else "API")
modified_cells = modified_cells[KEY_COLS + ["name", "column", "herkunft", "old", "new"]]

print(f"Neu hinzugekommen: {len(added_rows)} Zeilen")
print(f"Entfernt:          {len(removed_rows)} Zeilen")
print(f"Geänderte Zellen:  {len(modified_cells)} Zellen, "
      f"davon {(modified_cells['herkunft'] == 'manuell').sum()} in manuell gepflegten Spalten")

display(added_rows[KEY_COLS + ["name", "von", "bis"]])
display(removed_rows[KEY_COLS + ["name", "von", "bis"]])
display(modified_cells.head(20))


Neu hinzugekommen: 1 Zeilen
Entfernt:          0 Zeilen
Geänderte Zellen:  3 Zellen, davon 3 in manuell gepflegten Spalten


,id,name,von,bis
0,101,TEST Markt,,


,id,name,von,bis


,id,name,column,herkunft,old,new
0,109,Winter am THF,Fr,manuell,16:00 - 20:00,
1,109,Winter am THF,Sa,manuell,12:00 - 20:00,
2,109,Winter am THF,So,manuell,12:00 - 20:00,


In [8]:
# Übersicht pro Schlüssel 

if not modified_cells.empty:
    pivot = (
        modified_cells
        .assign(changed=1)
        .pivot_table(index=KEY_COLS, columns="column", values="changed", aggfunc="sum", fill_value=0)
        .reset_index()
    )
    display(pivot.head(20))
else:
    print("Keine geänderten Zellen.")


column,id,Fr,Sa,So
0,109,1,1,1


In [9]:
# Ergebnisse speichern
from datetime import datetime
import importlib.util
stamp = datetime.now().strftime("%d%m%y")

OUT_DIR = Path(f"reports/vergleich_report_{stamp}")
OUT_DIR.mkdir(exist_ok=True, parents=True)

# csv with ";" and utf-8-sig, so Excel opens it directly with umlauts
CSV_WRITE_KW = dict(index=False, sep=";", encoding="utf-8-sig")
added_rows.to_csv(OUT_DIR / "added_rows.csv", **CSV_WRITE_KW)
removed_rows.to_csv(OUT_DIR / "removed_rows.csv", **CSV_WRITE_KW)
modified_cells.to_csv(OUT_DIR / "modified_cells.csv", **CSV_WRITE_KW)

# Optional: alles in eine Excel-Datei schreiben (nur wenn openpyxl installiert ist)
if importlib.util.find_spec("openpyxl"):
    with pd.ExcelWriter(OUT_DIR / "report.xlsx", engine="openpyxl") as xw:
        added_rows.to_excel(xw, sheet_name="Added", index=False)
        removed_rows.to_excel(xw, sheet_name="Removed", index=False)
        modified_cells.to_excel(xw, sheet_name="ModifiedCells", index=False)

print("Ergebnisse gespeichert in:", OUT_DIR.resolve())


Ergebnisse gespeichert in: /Users/norahunger/Documents/GitHub/weihnachtsmarktkarte/data_processing/reports/vergleich_report_091126


In [10]:
# Update-Datei erzeugen (wird NICHT automatisch live gestellt)
# - alle Zeilen der Live-Datei bleiben unverändert (inkl. Handkorrekturen)
# - entfernte Märkte bleiben drin (siehe removed_rows.csv im Report)
# - geänderte Felder werden nicht übernommen (siehe modified_cells.csv im Report)
# - nur neue Märkte werden angehängt
# zum Live-Stellen die Datei von Hand nach public/markets_2026.csv kopieren

update = pd.concat([df_old, added_rows.reindex(columns=df_old.columns, fill_value="")],
                   ignore_index=True)

UPDATE_FILE = Path(f"{markets_old.stem}_update_{stamp}.csv")
update.to_csv(UPDATE_FILE, index=False, sep=",")

print(f"{len(df_old)} Märkte aus der Live-Datei + {len(added_rows)} neue = {len(update)} Märkte")
print(f"Nicht übernommen: {len(modified_cells)} geänderte Felder, {len(removed_rows)} entfernte Märkte bleiben drin")
print("Gespeichert:", UPDATE_FILE.resolve())


47 Märkte aus der Live-Datei + 1 neue = 48 Märkte
Nicht übernommen: 3 geänderte Felder, 0 entfernte Märkte bleiben drin
Gespeichert: /Users/norahunger/Documents/GitHub/weihnachtsmarktkarte/data_processing/markets_2026_update_091126.csv
